# Phase 2: Data Cleaning and Preparation

This notebook removes duplicates, handles missing values, standardizes dates, checks invalid sales values, creates the required columns, and saves the cleaned dataset.

## 1. Import libraries

In [ ]:
from pathlib import Path
import pandas as pd

## 2. Set the folder paths

In [ ]:
current_folder = Path.cwd()

if current_folder.name == 'notebooks':
    project_folder = current_folder.parent
else:
    project_folder = current_folder

raw_data_folder = project_folder / 'data' / 'raw'
processed_data_folder = project_folder / 'data' / 'processed'
processed_data_folder.mkdir(parents=True, exist_ok=True)

## 3. Load the datasets

In [ ]:
customers = pd.read_csv(raw_data_folder / 'Customers.csv', encoding='cp1252', keep_default_na=False, na_values=[''])
products = pd.read_csv(raw_data_folder / 'Products.csv', keep_default_na=False, na_values=[''])
sales = pd.read_csv(raw_data_folder / 'Sales.csv', keep_default_na=False, na_values=[''])
stores = pd.read_csv(raw_data_folder / 'Stores.csv', keep_default_na=False, na_values=[''])

print('Datasets loaded successfully.')

## 4. Remove duplicate records

In [ ]:
customers = customers.drop_duplicates().copy()
products = products.drop_duplicates().copy()
sales = sales.drop_duplicates().copy()
stores = stores.drop_duplicates().copy()

print('Duplicate records removed.')

## 5. Handle missing values

Blank delivery dates are kept empty because physical store purchases do not require delivery. The online store has no physical area, so its missing square meters value is changed to zero.

In [ ]:
online_store = stores['StoreKey'] == 0
stores.loc[online_store, 'Square Meters'] = stores.loc[online_store, 'Square Meters'].fillna(0)

print('Remaining missing values in Stores:')
display(stores.isna().sum())

print('Missing delivery dates kept as not applicable:', sales['Delivery Date'].isna().sum())

## 6. Standardize date columns

In [ ]:
sales['Order Date'] = pd.to_datetime(sales['Order Date'], errors='coerce')
sales['Delivery Date'] = pd.to_datetime(sales['Delivery Date'], errors='coerce')
customers['Birthday'] = pd.to_datetime(customers['Birthday'], errors='coerce')
stores['Open Date'] = pd.to_datetime(stores['Open Date'], errors='coerce')

print('Date columns converted to datetime format.')

## 7. Clean product cost and price columns

In [ ]:
products['Unit Cost USD'] = (
    products['Unit Cost USD']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

products['Unit Price USD'] = (
    products['Unit Price USD']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

display(products[['Unit Cost USD', 'Unit Price USD']].head())

## 8. Check and remove invalid sales values

In [ ]:
invalid_quantity = sales['Quantity'] <= 0
invalid_cost = products['Unit Cost USD'] < 0
invalid_price = products['Unit Price USD'] <= 0

print('Invalid sales quantities:', invalid_quantity.sum())
print('Invalid product costs:', invalid_cost.sum())
print('Invalid product prices:', invalid_price.sum())

sales = sales[~invalid_quantity].copy()
products = products[~invalid_cost & ~invalid_price].copy()

## 9. Combine the datasets

The tables are joined using their common keys so the cleaned file contains sales, product, customer, and store information.

In [ ]:
cleaned_sales = sales.merge(products, on='ProductKey', how='left')
cleaned_sales = cleaned_sales.merge(customers, on='CustomerKey', how='left')
cleaned_sales = cleaned_sales.merge(
    stores,
    on='StoreKey',
    how='left',
    suffixes=('_Customer', '_Store'),
)

print('Rows in cleaned dataset:', len(cleaned_sales))

## 10. Create sales and profit columns

In [ ]:
cleaned_sales['Revenue USD'] = cleaned_sales['Quantity'] * cleaned_sales['Unit Price USD']
cleaned_sales['Cost USD'] = cleaned_sales['Quantity'] * cleaned_sales['Unit Cost USD']
cleaned_sales['Gross Profit USD'] = cleaned_sales['Revenue USD'] - cleaned_sales['Cost USD']

## 11. Create the required derived columns

In [ ]:
cleaned_sales['Year'] = cleaned_sales['Order Date'].dt.year
cleaned_sales['Quarter'] = cleaned_sales['Order Date'].dt.quarter
cleaned_sales['Month'] = cleaned_sales['Order Date'].dt.month
cleaned_sales['Month Name'] = cleaned_sales['Order Date'].dt.month_name()
cleaned_sales['Day of Week'] = cleaned_sales['Order Date'].dt.day_name()

cleaned_sales['Gross Margin %'] = (
    cleaned_sales['Gross Profit USD'] / cleaned_sales['Revenue USD'] * 100
)

cleaned_sales['Sales Category'] = pd.qcut(
    cleaned_sales['Revenue USD'].rank(method='first'),
    q=3,
    labels=['Low', 'Medium', 'High'],
)

display(cleaned_sales[[
    'Order Date', 'Year', 'Quarter', 'Month', 'Month Name',
    'Day of Week', 'Gross Margin %', 'Sales Category'
]].head())

## 12. Save the cleaned dataset

Dates are saved in `YYYY-MM-DD` format.

In [ ]:
output_file = processed_data_folder / 'cleaned_sales.csv'

cleaned_sales.to_csv(
    output_file,
    index=False,
    date_format='%Y-%m-%d',
)

print('Cleaned dataset saved to:', output_file)